# 00_config — shared config for the GWAS of Maladaptive Recovery After Elective Orthopedic Surgery (All of Us / Verily Workbench)

Every later notebook starts by running this file: `%run 00_config.ipynb`.

## 🔒 COMPLIANCE (DUCC §3.1) — read before running anything
This session is driven through a browser automation that transmits notebook output to an external AI model.
**No individual-level (participant-level) data may ever be displayed.** Use `safe_show`, `safe_counts`, `round20`
below. Never call `df.head()`, never print rows, never display per-person genotypes/dates/MMEs. Model-visible
output = aggregates only, counts rounded to 20, cells <20 suppressed. All AI/ML on row-level data stays in this VM.

## 💸 COST
This notebook runs on a paid cloud environment. Pause when idle; **delete** the environment at end of session
(notebooks persist in the workspace, not on the disk).

In [ ]:
import os, subprocess
import pandas as pd

# ---- Google project (this workspace\'s GCP project; used for BigQuery billing) ----
GOOGLE_PROJECT = (os.environ.get("GOOGLE_PROJECT")
                  or os.environ.get("GOOGLE_CLOUD_PROJECT")
                  or os.environ.get("GCP_PROJECT"))

def wb_resolve(name):
    """Resolve a Verily Workbench *referenced resource* (by name) to its cloud id.
    Verily references the CDR/buckets by NAME (e.g. \'C2025Q4R6\'); this returns the
    fully-qualified BigQuery dataset (project.dataset) or gs:// path."""
    for args in (["wb", "resource", "resolve", "--name", name],
                 ["wb", "resolve", "--name", name]):
        try:
            out = subprocess.run(args, capture_output=True, text=True, timeout=90)
            val = (out.stdout or "").strip()
            if val:
                return val
        except Exception:
            pass
    return None

# ---- CDR (OMOP BigQuery dataset, cdrv9 = C2025Q4R6) ----
# Classic AoU exposes os.environ["WORKSPACE_CDR"]; Verily may not, so fall back to CLI resolve.
WORKSPACE_CDR = os.environ.get("WORKSPACE_CDR") or wb_resolve("C2025Q4R6")
PREP_CDR      = os.environ.get("WORKSPACE_CDR_PREP") or wb_resolve("prep_C2025Q4R6")

print("GOOGLE_PROJECT :", GOOGLE_PROJECT)
print("WORKSPACE_CDR  :", WORKSPACE_CDR)      # dataset id is metadata, safe to show
print("PREP_CDR       :", PREP_CDR)
# If any are None at runtime, resolve manually in a terminal:  wb resource resolve --name C2025Q4R6

In [ ]:
# ---- BigQuery query helper ----
from google.cloud import bigquery
_bq = bigquery.Client(project=GOOGLE_PROJECT)

def q(sql):
    """Run Standard SQL against the CDR -> DataFrame.
    Use {CDR} and {PREP} placeholders for the dataset ids."""
    sql = sql.replace("{CDR}", WORKSPACE_CDR or "").replace("{PREP}", PREP_CDR or "")
    return _bq.query(sql).result().to_dataframe()

In [ ]:
# ================= §3.1 COMPLIANCE HELPERS (use these, always) =================
MIN_CELL = 20  # All of Us dissemination threshold

def round20(n):
    """Round a count to nearest 20; 1-20 -> suppressed string."""
    try:
        n = int(n)
    except Exception:
        return n
    if n == 0:
        return 0
    if n <= MIN_CELL:
        return "<=20 (suppressed)"
    return int(round(n / 20.0) * 20)

def safe_show(df, name="df"):
    """Print ONLY shape + columns. NEVER prints rows (blocks individual-level egress)."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols  [rows hidden by DUCC policy]")
    print("columns:", list(df.columns))

def safe_n(df):
    """Dissemination-safe row count."""
    return round20(len(df))

def safe_counts(series):
    """value_counts with <20 suppressed and others rounded to 20."""
    return series.value_counts(dropna=False).apply(round20)

print("compliance helpers ready: round20(), safe_show(), safe_n(), safe_counts()")

In [ ]:
# ---- Genomics / Controlled-Tier bucket paths (ACAF + exome callsets, ancestry, QC) ----
GENOMICS_FOLDER   = wb_resolve("v9-genomics-folder")            # gs:// object/prefix
CONTROLLED_BUCKET = wb_resolve("vwb-aou-datasets-controlled-v9")  # gs:// bucket
print("GENOMICS_FOLDER   :", GENOMICS_FOLDER)
print("CONTROLLED_BUCKET :", CONTROLLED_BUCKET)
# Expected genomic substrates for this GWAS (confirm exact subpaths once with:  !gsutil -u $GOOGLE_PROJECT ls <path>):
#   - srWGS ACAF-threshold callset (Hail VDS/MatrixTable; PLINK/BGEN where provided) -> common-variant GWAS
#   - srWGS exome callset -> rare-variant burden (REGENIE burden / SKAT-O)
#   - aux/ancestry/ancestry_preds.tsv  (research_id, ancestry_pred[7 groups], pca_features[16 PCs], probabilities)
#   - aux/ sample-QC / relatedness / flagged-sample files (sex-concordance, contamination, coverage)

In [ ]:
# ---- Aggregate-only smoke test: confirms BigQuery connectivity ----
_n = q("SELECT COUNT(*) AS n FROM `{CDR}.person`")["n"].iloc[0]
print("person rows (rounded to 20):", round20(_n))

## Reuse in later notebooks
```python
%run 00_config.ipynb
df = q("SELECT COUNT(*) AS n FROM `{CDR}.condition_occurrence`")
print(round20(df['n'].iloc[0]))
```
Never surface raw rows. If you must inspect individual-level data, do it manually with the browser automation paused.